# Track 1 Smart Campus - prediction notebook (c22_distilled_compact)

Inference only: loads the trained artifact `model.pkl` (trained locally, see the training notebook) and predicts `energy_usage` for every row of the input file, in input order. All features are row-local, so shuffled rows / a missing `id` column are fine.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")

In [ ]:
# Feature engineering - identical to training (src/features.py)
import numpy as np
import pandas as pd

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BUILDINGS = ["ADM_A", "BUS_A", "BUS_B", "ENG_A", "ENG_B", "LEC_A", "LIB_A",
             "RES_A", "RES_B", "SCI_A", "SCI_B", "SPT_A"]
TYPES = ["Administration", "Business", "Engineering", "LectureHall", "Library",
         "Residential", "Science", "Sports"]
BUILDING_TYPE = {"ADM_A": "Administration", "BUS_A": "Business", "BUS_B": "Business",
                 "ENG_A": "Engineering", "ENG_B": "Engineering", "LEC_A": "LectureHall",
                 "LIB_A": "Library", "RES_A": "Residential", "RES_B": "Residential",
                 "SCI_A": "Science", "SCI_B": "Science", "SPT_A": "Sports"}
NUMERIC = ["previous_usage", "occupancy", "temperature", "humidity"]
# physically plausible ranges; values outside are treated as sensor errors -> missing
VALID = {"previous_usage": (0, 1000), "occupancy": (0, 10000), "temperature": (-20, 60), "humidity": (0, 100)}


def _num(s):
    return pd.to_numeric(s, errors="coerce").astype(float)


def clean_input(df):
    """Sanitise raw columns; anything unusable becomes NaN (handled downstream)."""
    out = pd.DataFrame(index=df.index)
    get = lambda c: df[c] if c in df.columns else pd.Series(np.nan, index=df.index)
    out["building_id"] = get("building_id").astype(str).str.strip().str.upper()
    out["building_type"] = get("building_type").astype(str).str.strip()
    hour = _num(get("hour"))
    out["hour"] = hour.where(hour.between(0, 23)).round()
    month = _num(get("month"))
    out["month"] = month.where(month.between(1, 12)).round()
    day = get("day_of_week").astype(str).str.strip().str[:3].str.lower()
    out["dow"] = day.map({d[:3].lower(): i for i, d in enumerate(DAYS)}).astype(float)
    for c in NUMERIC:
        v = _num(get(c))
        lo, hi = VALID[c]
        out[c] = v.where(v.between(lo, hi))
    return out


def make_features(df):
    """Tabular features for tree models (used by the GBM baseline)."""
    c = clean_input(df)
    out = pd.DataFrame(index=df.index)
    out["building"] = pd.Categorical(c["building_id"], categories=BUILDINGS)
    out["btype"] = pd.Categorical(c["building_type"], categories=TYPES)
    out["hour"] = c["hour"]
    out["dow"] = c["dow"]
    out["weekend"] = (c["dow"] >= 5).astype(float).where(c["dow"].notna())
    out["month"] = c["month"]
    for col in ["temperature", "humidity", "occupancy", "previous_usage"]:
        out[col] = c[col]
    return out


def _soft_building(c):
    """n x 12 building weights: one-hot if known, else uniform over buildings of the given type,
    else uniform over all buildings (=> prediction is the average building's expectation)."""
    n = len(c)
    W = np.zeros((n, len(BUILDINGS)))
    bid = c["building_id"].to_numpy(); bt = c["building_type"].to_numpy()
    for i in range(n):
        if bid[i] in BUILDINGS:
            W[i, BUILDINGS.index(bid[i])] = 1.0
        else:
            same = [j for j, b in enumerate(BUILDINGS) if BUILDING_TYPE[b] == bt[i]]
            W[i, same if same else slice(None)] = 1.0 / (len(same) if same else len(BUILDINGS))
    return W


def _soft_onehot(v, k):
    v = np.asarray(v, dtype=float)
    out = np.full((len(v), k), 1.0 / k)
    ok = ~np.isnan(v)
    out[ok] = 0.0
    out[np.where(ok)[0], v[ok].astype(int)] = 1.0
    return out


DEFAULT_SPEC = {"hours": "type", "weekend": "building",
                "slopes": {"previous_usage": "shared", "occupancy": "building",
                           "temperature": "building", "humidity": "shared"},
                "temp2": True, "season": True}


def linear_design(c, drop=(), spec=DEFAULT_SPEC):
    """Design matrix from a clean_input() frame. `drop` = numeric columns unavailable for this
    missingness-pattern sub-model. Categorical unknowns enter as expectations (soft one-hots)."""
    if spec.get("missing_spec") and set(drop) & {"occupancy", "previous_usage"}:
        # sub-models without occupancy/previous_usage must rebuild them from time x building -> richer design
        spec = {**spec, **spec["missing_spec"]}
    n = len(c)
    B = _soft_building(c)
    T = B @ np.array([[BUILDING_TYPE[b] == t for t in TYPES] for b in BUILDINGS], dtype=float)
    G = {"building": B, "type": T, "shared": np.ones((n, 1))}
    K = spec.get("hour_fourier", 0)
    if K:  # smooth daily profile: K harmonics (missing hour -> 0 = daily average)
        hh = c["hour"].to_numpy()
        H = np.concatenate([f(2 * np.pi * k * hh / 24)[:, None] for k in range(1, K + 1) for f in (np.sin, np.cos)], 1)
        H = np.nan_to_num(H, nan=0.0)
    else:
        H = _soft_onehot(c["hour"], 24)
    dow = c["dow"].to_numpy()
    we = np.where(np.isnan(dow), 2 / 7, (dow >= 5).astype(float))[:, None]
    cols = [B, (G[spec["hours"]][:, :, None] * H[:, None, :]).reshape(n, -1), G[spec["weekend"]] * we]
    if spec.get("hour_dev_scale"):  # per-hour deviations from the smooth profile, shrunk by a heavier
        # effective ridge penalty (alpha / scale^2) -> captures schedule steps (e.g. residential 08:00 / 21:00)
        Hd = _soft_onehot(c["hour"], 24) * spec["hour_dev_scale"]
        Gd = G[spec.get("hour_dev_group", "type")]
        cols.append((Gd[:, :, None] * Hd[:, None, :]).reshape(n, -1))
    if spec.get("weekend_hours"):  # separate daily shape on weekends
        Gw = G[spec["weekend_hours"]]
        cols.append((Gw[:, :, None] * (H * we)[:, None, :]).reshape(n, -1))
    for col in NUMERIC:
        if col not in drop:
            cols.append(G[spec["slopes"][col]] * c[col].to_numpy()[:, None])
    # extra shrinkable interactions: (numeric col, group, scale); effective penalty alpha/scale^2
    if spec.get("interactions"):
        hr = c["hour"].to_numpy()
        bands = np.stack([(hr <= 6), (hr >= 7) & (hr <= 11), (hr >= 12) & (hr <= 17), (hr >= 18)], 1).astype(float)
        bands[np.isnan(hr)] = 0.25
        we2 = np.concatenate([1 - we, we], 1)
        GX = {**G, "hourband": bands, "weekend": we2}
        for col, grp, scale in spec["interactions"]:
            if col in drop:
                continue
            if col == "temp_hinge":
                v = np.stack([np.maximum(c["temperature"].to_numpy() - k, 0) for k in (26.0, 28.0, 30.0)], 1)
                if "temperature" in drop:
                    continue
                cols.append((GX[grp][:, :, None] * v[:, None, :]).reshape(n, -1) * scale)
                continue
            cols.append(GX[grp] * c[col].to_numpy()[:, None] * scale)
    if spec.get("res_steps"):  # explicit residential day (08-21) / night schedule step indicators
        h = c["hour"].to_numpy(); res = T[:, TYPES.index("Residential")]
        day = np.where(np.isfinite(h), ((h >= 8) & (h < 21)).astype(float), 13 / 24)
        cols.append(np.stack([day * res, (1 - day) * res], 1))
    if spec.get("season", True):
        m = c["month"].to_numpy()
        sc = np.stack([np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)], 1)
        cols.append(np.nan_to_num(sc, nan=0.0))
    if spec.get("temp2", True) and "temperature" not in drop:
        cols.append(((c["temperature"].to_numpy() - 28.0) ** 2)[:, None])
    return np.concatenate(cols, 1)


def pattern_keys(c):
    miss = c[NUMERIC].isna().to_numpy()
    return [tuple(col for col, mm in zip(NUMERIC, row) if mm) for row in miss]


def predict_linear(art, df):
    """Pattern-submodel linear prediction: each row uses the model fitted without its missing
    numeric columns. art = {"coefs": {pattern: (coef, intercept)}, "spec": ..., "clip": (lo, hi)}."""
    c = clean_input(df)
    keys = pattern_keys(c)
    preds = np.empty(len(c))
    for k in set(keys):
        idx = np.array([kk == k for kk in keys])
        coef, icpt = art["coefs"][k]
        preds[idx] = linear_design(c[idx], drop=set(k), spec=art["spec"]) @ coef + icpt
    return np.clip(preds, *art["clip"])

In [ ]:
artifact = joblib.load("model.pkl")  # uploaded next to this notebook
test = pd.read_csv(TEST_PATH)

def _dev_keys(F, level):
    cols = [F[x].cat.codes.to_numpy() if hasattr(F[x], "cat") else F[x].to_numpy() for x in level]
    return list(zip(*cols))

def add_dev(F, tables, levels, skip):
    """Append deviation features (fixed order: level, then column) - identical in training and inference."""
    G = F.copy()
    for i, level in enumerate(levels):
        keys = _dev_keys(F, level)
        for c in ["occupancy", "previous_usage", "temperature", "humidity"]:
            if c != skip:
                mu = np.array([tables[i][c].get(k, np.nan) for k in keys], dtype=float)
                G[f"dev{i}_{c}"] = F[c].to_numpy(dtype=float) - mu
    return G

def add_cyclic(F):
    """sin/cos of hour and month for the imputers (appended after 'dom')."""
    G = F.copy(); h = G["hour"].to_numpy(dtype=float); m = G["month"].to_numpy(dtype=float)
    G["h_sin"], G["h_cos"] = np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24)
    G["m_sin"], G["m_cos"] = np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)
    return G

import lightgbm as lgb

# Known building IDs determine their type. All inputs remain row-local.
_raw = test.copy()
_clean = clean_input(_raw)
_type_names = {t.lower(): t for t in TYPES}
_types = _clean["building_type"].str.lower().map(_type_names)
_known = _clean["building_id"].map(BUILDING_TYPE)
_raw["building_type"] = _known.fillna(_types).to_numpy()

F = make_features(_raw)
F["dom"] = artifact["domain"]
if artifact["cyclic"]:
    F = add_cyclic(F)
X = _raw.copy()
_filled = {}
for col in artifact["impute_cols"]:
    _miss = F[col].isna().to_numpy()
    _filled[col] = np.zeros(len(F), dtype=bool)
    X[col] = F[col].to_numpy()
    if not _miss.any():
        continue
    try:
        _xm = add_dev(F, artifact["dev_tables"], artifact["dev_levels"], col).drop(columns=[col])[_miss]
        _values = np.mean([lgb.Booster(model_str=s).predict(_xm)
                           for s in artifact["imputers"][col]], axis=0)
        # The same physical bounds as clean_input; unusable fills stay missing.
        _lo, _hi = VALID[col]
        _ok = np.isfinite(_values) & (_values >= _lo) & (_values <= _hi)
        _positions = np.flatnonzero(_miss)
        X.iloc[_positions[_ok], X.columns.get_loc(col)] = _values[_ok]
        _filled[col][_positions[_ok]] = True
    except Exception as _error:
        print('WARNING: imputer fallback for', col, repr(_error))

try:
    preds = np.asarray(predict_linear(artifact["linear"], X), dtype=float).copy()
    _bid = clean_input(_raw)["building_id"].to_numpy()
    _complete = clean_input(X)[NUMERIC].notna().all(axis=1).to_numpy()
    for _col, _k in artifact["mnar_kappa"].items():
        _m = _filled[_col] & _complete
        if _k == 0 or not _m.any():
            continue
        _h = artifact["mnar_step"][_col]
        _up = X.loc[_m].copy(); _dn = X.loc[_m].copy()
        _up[_col] = _up[_col] + _h; _dn[_col] = _dn[_col] - _h
        _slope = (predict_linear(artifact["linear"], _up) - predict_linear(artifact["linear"], _dn)) / (2 * _h)
        _sd = np.array([artifact["mnar_sd"][_col].get(b, artifact["mnar_sd_default"][_col]) for b in _bid[_m]])
        preds[_m] = preds[_m] + np.nan_to_num(_k * _slope * _sd)
except Exception as _error:
    print('WARNING: energy model fallback:', repr(_error))
    try:
        preds = predict_linear(artifact["linear"], _raw)
    except Exception as _fallback_error:
        print('WARNING: mean fallback:', repr(_fallback_error))
        preds = np.full(len(_raw), artifact["fallback_mean"], dtype=float)

preds = np.asarray(preds, dtype=float).ravel()
preds = np.where(np.isfinite(preds), preds, artifact["fallback_mean"])

preds = np.asarray(preds, dtype=float).ravel()
assert len(preds) == len(test) and np.isfinite(preds).all()

In [ ]:
# platform input has no id column (template) -> 'prediction' only; with ids -> id,prediction
out = pd.DataFrame({"prediction": preds})
if "id" in test.columns:
    out.insert(0, "id", test["id"].to_numpy())
out.to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")